# 08 — Phân tích lỗi SVM
  Xem lỗi theo mệnh giá và các ví dụ ảnh như phần KNN. Điểm SVM không phải xác suất.
  Các ảnh train gần chỉ dùng đối chiếu; dự đoán vẫn do SVM đưa ra.

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_SVM_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_SVM/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('SVM', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'SVM', 'test_final/model.joblib')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224

## 1. Nạp mô hình và kiểm tra dự đoán

In [ ]:
import joblib
from src.experiments.figures import error_tables, select_cases, error_gallery, svm_reference_gallery
TEST = RESULTS_ROOT/'test_final'
OUTPUT = RESULTS_ROOT/'error_analysis'
OUTPUT.mkdir(parents=True, exist_ok=True)
lock = json.loads((RESULTS_ROOT/'selection/locked_svm_config.json').read_text())
X_train, _, train_manifest = load_features(RESULTS_ROOT/lock['feature_folder'], 'train')
X_test = np.load(TEST/'X_test.npy', mmap_mode='r')
predictions = pd.read_csv(TEST/'predictions_test.csv')
model = joblib.load(TEST/'model.joblib')
if not np.array_equal(model.predict(X_test), predictions.y_pred_vnd):
    raise ValueError('Mô hình và bảng dự đoán không khớp.')
decision = model.decision_function(X_test)
ranked = np.sort(decision, axis=1)
predictions['score_gap'] = ranked[:, -1]-ranked[:, -2]
predictions['test_index'] = np.arange(len(predictions))
for i, value in enumerate(model.classes_):
    predictions[f'score_{value}'] = decision[:, i]
predictions.to_csv(OUTPUT/'all_test_decisions.csv', index=False)

## 2. Lỗi theo lớp và các cặp nhầm

In [ ]:
per_class, directed = error_tables(predictions, OUTPUT)
display(per_class[['denomination_vnd', 'support', 'fn', 'f1']])
display(directed.head(10))

## 3. Chọn và xem ảnh
  Ưu tiên các chiều nhầm phổ biến, lỗi chênh mệnh giá lớn và ảnh đúng để đối chiếu.
  Khoảng cách giữa hai điểm cao nhất chỉ dùng chọn ví dụ, không phải độ tin cậy.

In [ ]:
cases = select_cases(predictions)
cases.to_csv(OUTPUT/'selected_cases.csv', index=False)
display(cases[['case_id', 'relative_path', 'y_true_vnd', 'y_pred_vnd', 'selection_reason']])
error_gallery(cases, DATA_ROOT, OUTPUT, 'SVM', decision)
svm_reference_gallery(cases, train_manifest, X_train, X_test, DATA_ROOT, OUTPUT)

## 4. Ghi nhận xét và tải kết quả
  Viết điều thấy trên ảnh trước khi nêu giả thuyết nguyên nhân. Không đổi cấu hình theo test.

In [ ]:
notes = cases[['case_id', 'relative_path']].copy()
notes['nhan_xet'] = ''
notes.to_csv(OUTPUT/'ghi_chu_anh.csv', index=False)
import shutil
archive = shutil.make_archive(str(RESULTS_ROOT.parent/'CS231_SVM_RESULTS'), 'zip',
                              root_dir=RESULTS_ROOT.parent, base_dir=RESULTS_ROOT.name)
print('Kết quả:', archive)